# 🚗 Car Price Prediction using Machine Learning
### CodeAlpha Data Science Internship — Task 3

**Objective:** Build and benchmark regression models (Linear Regression vs. Random Forest Regressor) to estimate car resale valuations based on vehicle specifications, age, and wear.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Load Data & Feature Engineering

In [ ]:
df = pd.read_csv(os.path.join('..', 'data', 'car data.csv'))
df.columns = df.columns.str.strip()
# Derive Car_Age relative to 2020
df['Car_Age'] = 2020 - df['Year']
df_clean = df.drop(columns=['Car_Name', 'Year'], errors='ignore')
df_encoded = pd.get_dummies(df_clean, drop_first=True)
df_encoded.head()

## 2. Train/Test Split

In [ ]:
X = df_encoded.drop(columns=['Selling_Price'])
y = df_encoded['Selling_Price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train set: {X_train.shape[0]} rows | Test set: {X_test.shape[0]} rows')

## 3. Model Training & Comparison

In [ ]:
lr = LinearRegression().fit(X_train, y_train)
rf = RandomForestRegressor(n_estimators=100, random_state=42).fit(X_train, y_train)

lr_preds = lr.predict(X_test)
rf_preds = rf.predict(X_test)

print(f'Linear Regression -> R2: {r2_score(y_test, lr_preds):.4f} | RMSE: {np.sqrt(mean_squared_error(y_test, lr_preds)):.3f}')
print(f'Random Forest     -> R2: {r2_score(y_test, rf_preds):.4f} | RMSE: {np.sqrt(mean_squared_error(y_test, rf_preds)):.3f}')

## 4. Visualizations: Feature Importance & Actual vs Predicted

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Feature Importance
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
sns.barplot(ax=axes[0], x=importances.values, y=importances.index, hue=importances.index, legend=False, palette='Blues_r')
axes[0].set_title('Feature Importance (Random Forest)', fontsize=13, fontweight='bold')

# Actual vs Predicted
axes[1].scatter(y_test, rf_preds, alpha=0.7, color='navy', edgecolors='k')
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[1].set_title('Actual vs Predicted Prices (Lakhs)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Actual')
axes[1].set_ylabel('Predicted')

plt.tight_layout()
plt.show()